#### Import modules

In [1]:
# Simple import setup
import sys
import os
sys.path.append('..')  # Add parent directory to path

# Import all necessary modules
import pandas as pd
import math
import matplotlib.pyplot as plt

# Import from the new modular structure
from src_code import (
    pf_enviroment, initialize_powerfactory, activate_project, list_and_select_study_case,
    list_and_activate_operation_scenario, run_contingency_analysis, process_cargabilidad, optimize_generators_for_substations, show_iteration_details,
    create_static_generator, calculate_power_limits, delete_generator, update_generator_power, cleanup_existing_generator, cleanup_all_test_generators
    )

### PowerFactory environment definition ------------

#### Initialize PowerFactory

In [2]:
# Añadir la ruta del entorno de PowerFactory
dig_path = r'C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9'
pf_enviroment(dig_path)

PowerFactory environment initialized with path: C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9


#### Get PowerFactory application

In [3]:
# Initialize PowerFactory application
app = initialize_powerfactory()

PowerFactory application connected successfully!


#### Activate project

In [4]:
project_name = "39 Bus New England System"
project = activate_project(app, project_name)

Proyecto '39 Bus New England System' activado con exito.


#### Activate study case

In [5]:
# Activate the desired study case
study_case_name = "1. Power Flow"  # Replace with your specific case name
active_study_case = list_and_select_study_case(app, study_case_name)

List of study cases:
2.1 Simulation Fault Bus 16 Stable
2.2 Simulation Fault Bus 16 Unstable
2.3 Simulation Fault Bus 31 Stable
2.4 Simulation Fault Bus 31 Unstable
2.5 Simulation Fault Line 2-3 Stable
2.6 Simulation Fault Line 2-3 Unstable
3. Small Signal Analysis (Eigenvalues)
4. EMT Simulation Fault Bus 03
1. Power Flow
Study case '1. Power Flow' activated.


#### Activate operation scenario

In [6]:
operation_scenario_name = "Basic Load Flow"
active_scenario = list_and_activate_operation_scenario(app, operation_scenario_name)

List of operation scenarios:
EMT
Basic Load Flow
Operation scenario 'Basic Load Flow' activated.


### Run contingency analysis ---------------

#### Get network data

In [7]:
project = app.GetActiveProject()
network_data = project.GetContents('Network Model.IntPrjfolder\\Network Data', 1)[0]
hoja = 'Grid'

#### Get buses

In [8]:
# buses = app.GetCalcRelevantObjects('*.ElmTerm')
# buses = [bus.loc_name for bus in buses]

#### Run Contingency analysis Generator Optimizer

In [ ]:
# Main execution
print("🚀 INICIANDO ANÁLISIS DE CONTINGENCIA CON OPTIMIZACIÓN DE GENERADORES")
print("="*80)

# Clean up any existing generators first
print("Limpiando generadores existentes...")
cleanup_all_test_generators(app)

# Define substations to analyze
substations = [
    "Bus 03",
    "Bus 04", 
    "Bus 07",
    "Bus 08",
    "Bus 09"
]

# Parameters
initial_potencia = 1  # MW
factor_potencia = 0.95
max_cargabilidad = 110  # %
threshold_inconvergence = 10  # %

print(f"Subestaciones a analizar: {substations}")
print(f"Potencia inicial: {initial_potencia} MW")
print(f"Factor de potencia: {factor_potencia}")
print(f"Límite de cargabilidad: {max_cargabilidad}%")
print(f"Umbral de inconvergencia: {threshold_inconvergence}%")

# Run optimization
df_results = optimize_generators_for_substations(
    app=app,
    network_data=network_data,
    hoja=hoja,
    substations=substations,
    initial_potencia=initial_potencia,
    factor_potencia=factor_potencia,
    max_cargabilidad=max_cargabilidad,
    threshold_inconvergence=threshold_inconvergence
)

print(f"\n✅ ANÁLISIS COMPLETADO")
print(f"Total de subestaciones analizadas: {len(df_results)}")
if not df_results.empty:
    print(f"Potencia máxima total posible: {df_results['Potencia_Maxima'].sum()} MW")
    print(f"Promedio de potencia por subestación: {df_results['Potencia_Maxima'].mean():.2f} MW")


🚀 INICIANDO ANÁLISIS DE CONTINGENCIA CON OPTIMIZACIÓN DE GENERADORES
Limpiando generadores existentes...
Limpiando todos los generadores de prueba existentes...
Encontrado generador de prueba: 'Gen_Estatico_Bus 06'
Eliminando generador: 'Gen_Estatico_Bus 06'
Encontrado cubículo de prueba: 'Cubicle_Gen_Bus 06'
Eliminando cubículo: 'Cubicle_Gen_Bus 06'
Limpieza completada. Eliminados 1 generadores y 1 cubículos.
Subestaciones a analizar: ['Bus 03', 'Bus 04', 'Bus 05', 'Bus 07', 'Bus 08', 'Bus 09']
Potencia inicial: 1 MW
Factor de potencia: 0.95
Límite de cargabilidad: 110%
Umbral de inconvergencia: 10%
Optimizando generador para la subestación 'Bus 03'.
Creando generador estático en la barra 'Bus 03' con potencia activa 1 MW y factor de potencia 0.95.
Límites calculados: S = 1.05 MVA, Q_max = 0.33 MVar, Q_min = -0.33 MVar
Buscando hoja 'Grid' en 'Network Data'.
Debug: Found folder: Diagrams (type: IntPrjfolder)
Debug: Found folder: Network Data (type: IntPrjfolder)
Debug: Found folder: O

KeyboardInterrupt: 